# TimesFM — Explainability & Two-Regime Comparison

TimesFM (Google) as the **third** SentiSense forecaster, alongside XGBoost + LSTM.
Reuses the `sentisense` package, the shared `models.backtest` scorers, and
`scripts/pipeline_compare.build_leaderboard` — **no metric or plot is re-invented**.

**Forecast→direction bridge:** TimesFM forecasts the daily **log-return**; we map
`sign(forecast)`→Up/Down and score the signed magnitude via `forecast_to_proba` on the
SAME `metrics_at` as the classifiers (apples-to-apples).

**Two regimes (first-class arms):** `CUT` = data ≤ 2023-10-07 (regime-break guard),
`FULL` = entire timeline. The CUT-vs-FULL delta measures the regime break.

**Convention:** visualizations show ALL dates in the DB; each model only *considers*
data within its regime.

## Prerequisites
```bash
uv sync --extra ml --extra finance --extra notebook
uv pip install 'timesfm[torch]'   # or: git clone .../timesfm && uv pip install -e '.[torch]'
# HF_TOKEN in env if the checkpoint is gated. Needs GPU.
```


## 0. Setup

In [ ]:
from __future__ import annotations
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sentisense  # loads .env
from sentisense.db import get_engine
from sentisense.constants import CUTOFF_DATE, TA125_CSV
from sentisense.features import build_datasets
from sentisense.models.backtest import (forecast_to_proba, direction_metrics,
                                         next_day_returns, strategy_stats, directions_from_price)
from sentisense.models.timesfm_forecaster import (load_timesfm, make_forecast_fn,
                                                  walk_forward_directions, finetune_on_train, CONTEXT_LEN)

sns.set_theme(style="whitegrid", context="notebook")
engine = get_engine()
FAR_FUTURE = pd.Timestamp("2100-01-01")
REGIMES = {"CUT": pd.Timestamp(CUTOFF_DATE), "FULL": FAR_FUTURE}

# TA-125 close (all dates) — same parse as the analysis notebook.
ta = pd.read_csv(TA125_CSV); ta["Date"] = pd.to_datetime(ta["Date"], errors="coerce")
price = (ta.dropna(subset=["Date"]).set_index("Date").sort_index()
         ["Price"].astype(str).str.replace(",", "", regex=False).astype(float))
print("price span:", price.index.min().date(), "→", price.index.max().date())

# Load TimesFM once (guarded — prints the install hint if the extra is absent).
try:
    TFM = load_timesfm(CONTEXT_LEN)
except RuntimeError as e:
    print("TimesFM unavailable:\n", e); TFM = None


In [ ]:
# Helper: walk-forward TimesFM scores for a regime (reuses the package harness).
def regime_returns(cutoff):
    p = price[price.index <= pd.Timestamp(cutoff)].sort_index()
    return np.log(p / p.shift(1)).dropna()

def regime_scores(cutoff, *, covariate_frame=None, finetune_returns=None):
    if TFM is None:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    r = regime_returns(cutoff)
    split = int(len(r) * 0.85)
    test_index = r.index[split:]
    if finetune_returns is not None:
        finetune_on_train(TFM, r.iloc[:split])
    fn = make_forecast_fn(TFM, covariate_frame=covariate_frame)
    return walk_forward_directions(r, test_index, fn, context_len=CONTEXT_LEN)


## 1. Forecast vs actual + buy-only overlay (per regime)

Walk-forward TimesFM (zero-shot) over the held-out window. Top: predicted vs realised
next-day return. Then the **buy-only overlay** on the scores — model decisions on the
held-out window combined with a forced BUY on every post-2023-10-07 day, scored vs the
real direction (the project convention).

In [ ]:
scores_cut, labels_cut = regime_scores(REGIMES["CUT"])
if len(scores_cut):
    r = regime_returns(REGIMES["CUT"]); fc = (scores_cut - 0.5)  # signed proxy of forecast sign
    nxt = pd.Series(next_day_returns(price, scores_cut.index), index=scores_cut.index)
    fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax[0].plot(nxt.index, nxt.values, lw=0.8, label="actual next-day return")
    ax[0].axhline(0, c="gray", ls=":"); ax[0].set_title("Held-out window: actual next-day returns (CUT)"); ax[0].legend()
    ax[1].plot(scores_cut.index, scores_cut.values, lw=0.8, c="#7c3aed")
    ax[1].axhline(0.5, c="gray", ls=":"); ax[1].set_title("TimesFM direction score (>0.5 = Up)"); plt.show()
    m = direction_metrics(scores_cut.to_numpy(), labels_cut.to_numpy())
    print("CUT zero-shot:", {k: round(v, 4) for k, v in m.items()})
else:
    print("TimesFM unavailable — skipping.")


In [ ]:
# Buy-only overlay: held-out model decisions + forced BUY after 2023-10-07.
if len(scores_cut):
    from sklearn.metrics import accuracy_score, confusion_matrix
    anchor = pd.Timestamp("2023-10-07")
    d_all = directions_from_price(price)
    later = d_all[d_all.index > anchor]
    dec = np.concatenate([(scores_cut.to_numpy() > 0.5).astype(int), np.ones(len(later), int)])
    truth = np.concatenate([labels_cut.to_numpy().astype(int), later.to_numpy().astype(int)])
    cm = confusion_matrix(truth, dec, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Down", "Up"], yticklabels=["Down", "Up"])
    ax.set_title(f"Buy-overlay confusion — acc {accuracy_score(truth, dec):.3f}"); plt.show()


## 2. Is the predicted magnitude informative (not just the sign)?

Calibration of the direction score (binned predicted-up-prob vs realised up-rate) +
the residual/score distribution. If only the sign matters, calibration is flat.

In [ ]:
if len(scores_cut):
    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    frac, mean_pred = calibration_curve(labels_cut.to_numpy(), scores_cut.to_numpy(), n_bins=8, strategy="quantile")
    ax[0].plot(mean_pred, frac, "o-"); ax[0].plot([0, 1], [0, 1], "--", c="gray")
    ax[0].set_title("Sign-decision calibration (CUT)"); ax[0].set_xlabel("pred up-prob"); ax[0].set_ylabel("observed up-rate")
    ax[1].hist(scores_cut.to_numpy(), bins=30, color="#0ea5e9"); ax[1].axvline(0.5, c="red", ls=":")
    ax[1].set_title("Direction-score distribution"); plt.show()


## 3. Covariate ablation — does TimesFM use the news signal?

TimesFM with sentiment features as XReg covariates vs univariate (same window, CUT).
If the installed build lacks covariate support the cov form falls back to univariate
(logged) — then the two bars match by construction.

In [ ]:
if TFM is not None:
    mt_cut, _ = build_datasets(engine, cutoff=REGIMES["CUT"])
    news = [c for c in mt_cut.columns if c.startswith(("mean_", "ix_"))]
    r = regime_returns(REGIMES["CUT"])
    cov = mt_cut[news].reindex(r.index).fillna(0.0) if news else None
    s_nocov, l_nocov = regime_scores(REGIMES["CUT"])
    s_cov, l_cov = regime_scores(REGIMES["CUT"], covariate_frame=cov)
    abl = pd.DataFrame({
        "no covariates": direction_metrics(s_nocov.to_numpy(), l_nocov.to_numpy()),
        "with covariates": direction_metrics(s_cov.to_numpy(), l_cov.to_numpy()),
    }).T[["roc_auc", "accuracy", "mcc"]]
    display(abl)
    abl.plot.bar(figsize=(8, 3), title="Covariate ablation (CUT)"); plt.ylim(0.4, 0.7); plt.show()


## 4. Regime comparison — CUT vs FULL

Hypothesis: the 2023-10-07 regime break degrades FULL-trained generalization. Same
metric both regimes; show the delta.

In [ ]:
rows = {}
for name, cut in REGIMES.items():
    s, l = regime_scores(cut)
    if len(s):
        rows[name] = direction_metrics(s.to_numpy(), l.to_numpy())
if rows:
    reg = pd.DataFrame(rows).T[["roc_auc", "accuracy", "f1", "mcc"]]
    display(reg)
    reg[["roc_auc", "accuracy"]].plot.bar(figsize=(7, 3), title="TimesFM zero-shot: CUT vs FULL"); 
    plt.axhline(0.5, c="gray", ls=":"); plt.ylim(0.4, 0.7); plt.show()
    if {"CUT", "FULL"} <= set(reg.index):
        print("ΔROC-AUC (FULL − CUT):", round(reg.loc["FULL", "roc_auc"] - reg.loc["CUT", "roc_auc"], 4))


## 5. Leaderboard — all models × both regimes (out-of-sample)

Reuses `scripts/pipeline_compare.build_leaderboard` (same `metrics_at` + backtest util)
so TimesFM sits on one board with XGBoost, LSTM, and Buy&Hold.

In [ ]:
sys.path.insert(0, "scripts")
from pipeline_compare import build_leaderboard
board = build_leaderboard(["CUT", "FULL"], use_timesfm=(TFM is not None))
display(board.round(4))


In [ ]:
# Walk-forward equity curves on one axis (held-out window, CUT).
if len(scores_cut):
    fig, ax = plt.subplots(figsize=(13, 4))
    for name, (s, l) in {"TimesFM": (scores_cut, labels_cut)}.items():
        nxt = next_day_returns(price, s.index)
        eq = np.cumprod(1 + (s.to_numpy() > 0.5).astype(float) * nxt)
        ax.plot(s.index, eq, label=f"{name} (Sharpe {strategy_stats((s.to_numpy()>0.5), nxt)['sharpe']:.2f})")
    bh_nxt = next_day_returns(price, scores_cut.index)
    ax.plot(scores_cut.index, np.cumprod(1 + bh_nxt), label="Buy & Hold", c="gray")
    ax.set_title("Held-out equity (CUT)"); ax.legend(); plt.show()


## 6. Notes
- TimesFM is **zero-shot-friendly** on this small daily series — contrast with LSTM overfit.
- The honest read still applies: near-chance direction (see §4 / leaderboard) means the
  edge, if any, is in magnitude/regime, not the raw sign. Compare CUT vs FULL for the
  regime-break effect, and the covariate ablation (§3) for whether news helps.
- All numbers come from the shared `metrics_at` + `models.backtest`; the leaderboard is
  `pipeline_compare.build_leaderboard` — nothing re-implemented here.
